# 1) Title and Goals

This project compares several classifiers on the MNIST handwritten digit dataset and asks: **where does each model 'look' to make its decision?**

We use a single, simple rule for visual explanations: **brighter = mattered more** (occlusion-based importance). The notebook focuses on: 

- Easy wins where the CNN is right and others fail.
- Tricky cases where many models struggle.

**Caveat:** Statements such as "CNN almost always wins because convolutional filters learn local stroke features" are supported by the visualizations in this notebook but are **not** a proof of generality. They are empirical observations on MNIST and should be interpreted cautiously.


In [ ]:
# 2) Setup and Imports
import time
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibility seeds
np.random.seed(42)
import random
random.seed(42)
tf.random.set_seed(42)

print('numpy, sklearn, tensorflow versions:', np.__version__, sklearn.__version__, tf.__version__)


In [ ]:
# 3) Load MNIST Data
mnist = fetch_openml(name='mnist_784', version=1, as_frame=False, parser='auto')
X = mnist.data.astype('float32') / 255.0
y = mnist.target.astype(int)
X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]
X_train_img = X_train.reshape(-1,28,28,1)
X_test_img = X_test.reshape(-1,28,28,1)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)


In [ ]:
# 4) Define Models
models_sklearn = {
    'KNN': KNeighborsClassifier(n_neighbors=3),
    'DecisionTree': DecisionTreeClassifier(random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'NaiveBayes': GaussianNB(),
    'MLP': MLPClassifier(random_state=42, max_iter=300),
}

# Compact CNN (Keras)
def build_cnn():
    m = keras.Sequential([
        layers.Input(shape=(28,28,1)),
        layers.Conv2D(32, 3, activation='relu', name='conv1'),
        layers.MaxPooling2D(name='pool1'),
        layers.Conv2D(64, 3, activation='relu', name='conv2'),
        layers.MaxPooling2D(name='pool2'),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.25),
        layers.Dense(10, activation='softmax'),
    ])
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

cnn = build_cnn()


In [ ]:
# 5) Train Models
print('Training scikit-learn models...')
for name, clf in models_sklearn.items():
    print('Training', name)
    clf.fit(X_train, y_train)
print('Scikit-learn models trained.')

print('\nTraining CNN...')
callbacks = [keras.callbacks.EarlyStopping(patience=2, monitor='val_accuracy', restore_best_weights=True)]
cnn.fit(X_train_img, y_train, validation_split=0.1, epochs=8, batch_size=128, callbacks=callbacks, verbose=2)


In [ ]:
# 6) Evaluate Accuracy
preds = {name: clf.predict(X_test) for name, clf in models_sklearn.items()}
accs = {name: accuracy_score(y_test, preds[name]) for name in models_sklearn}
proba_cnn = cnn.predict(X_test_img, verbose=0)
preds['CNN'] = np.argmax(proba_cnn, axis=1)
accs['CNN'] = accuracy_score(y_test, preds['CNN'])
print('--- Model Test Accuracies ---')
for name in ['KNN','DecisionTree','RandomForest','NaiveBayes','MLP','CNN']:
    print(f"{name:<13} accuracy: {accs[name]:.2%}")

# Confusion matrix for CNN
import seaborn as sns
cm = confusion_matrix(y_test, preds['CNN'])
plt.figure(figsize=(6,5))
sns.heatmap(cm, cmap='Blues', square=True)
plt.title('CNN Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()


In [ ]:
# 7) Explainability helpers: predict_proba_any and occlusion
def predict_proba_any(model, x_img, model_name=None):
    # x_img: 2D (28,28)
    if model_name is None:
        model_name = getattr(model, '__class__', None)
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(x_img.reshape(1, -1))[0]
    x = x_img.reshape(1,28,28,1).astype('float32')
    return model.predict(x, verbose=0)[0]

def simple_importance_map_any(model, x_img, window=4, stride=2, model_name=None, top_quantile=0.90):
    """Occlusion-based importance map for any model.
    - window, stride: control patch size and sliding step.
    - top_quantile: which fraction of the importance map is considered 'standout'.
    Returns (importance_map, predicted_class, base_confidence, mask)
    """
    H, W = x_img.shape
    base_proba = predict_proba_any(model, x_img, model_name=model_name)
    pred = int(np.argmax(base_proba)); base = float(base_proba[pred])
    # For KNN/NB, use larger window / smaller stride for more stable occlusion
    if model_name in {'KNN', 'NaiveBayes'}:
        window, stride = max(window, 7), min(stride, 2)
    imp = np.zeros((H, W), dtype=float); cnt = np.zeros((H, W), dtype=float)
    for r in range(0, H - window + 1, stride):
        for c in range(0, W - window + 1, stride):
            xb = x_img.copy(); xb[r:r+window, c:c+window] = 0.0
            xw = x_img.copy(); xw[r:r+window, c:c+window] = 1.0
            drop_b = max(0.0, base - float(predict_proba_any(model, xb, model_name=model_name)[pred]))
            drop_w = max(0.0, base - float(predict_proba_any(model, xw, model_name=model_name)[pred]))
            drop = max(drop_b, drop_w)
            imp[r:r+window, c:c+window] += drop; cnt[r:r+window, c:c+window] += 1.0
    cnt[cnt==0] = 1.0
    imp /= cnt
    q = np.quantile(imp, top_quantile)
    mask = (imp >= q) & (imp > 0)
    return imp, pred, base, mask

def nb_pixel_contrib(gnb, x_img, pred_class):
    mu = gnb.theta_[pred_class].reshape(28,28)
    var_attr = getattr(gnb, 'var_', None) or getattr(gnb, 'sigma_', None)
    var = var_attr[pred_class].reshape(28,28) + 1e-9
    # log-probability contribution per pixel (ignoring additive constants)
    return -0.5 * ((x_img - mu)**2 / var) - 0.5 * np.log(var)


In [ ]:
# 8) Visualization helpers
def show_row(idx, x_img, y_true, panels, cohort_title):
    fig, axes = plt.subplots(1, 6, figsize=(19, 3.8))
    axes[0].imshow(x_img, cmap='gray'); axes[0].set_title(f"{cohort_title}\nOriginal\nTrue: {y_true}"); axes[0].axis('off')
    for j, (name, overlay, pred, conf, ok, mask) in enumerate(panels, start=1):
        axes[j].imshow(x_img, cmap='gray', alpha=0.8)
        if overlay is not None:
            vmax = np.percentile(overlay, 99.0) if np.max(overlay)>0 else 1.0
            axes[j].imshow(overlay, cmap='inferno', alpha=0.6, vmin=0.0, vmax=vmax)
        title_color = 'green' if ok else 'red'
        axes[j].set_title(f"{name}\nPred: {pred} • {conf:.0%}", fontsize=11, color=title_color)
        axes[j].axis('off')
    plt.tight_layout(); plt.show()

def pick(mask, k, seed=0):
    """Deterministically pick examples from a boolean mask.
    Returns the first k indices satisfying mask. If none, fall back to random selection using seed.
    """
    idxs = np.where(mask)[0]
    if idxs.size == 0:
        print('Warning: No examples found for mask. Picking random.')
        rng = np.random.RandomState(seed)
        return rng.choice(X_test.shape[0], size=min(k, X_test.shape[0]), replace=False)
    return idxs[:k]


In [ ]:
# 9) Select Cohorts and Render comparisons
proba_cnn = cnn.predict(X_test_img, verbose=0)
preds = {name: models_sklearn[name].predict(X_test) for name in models_sklearn}
accs = {name: accuracy_score(y_test, preds[name]) for name in models_sklearn}
preds['CNN'] = np.argmax(proba_cnn, axis=1); accs['CNN'] = accuracy_score(y_test, preds['CNN'])
corrects = {name: (preds[name]==y_test) for name in preds}
all_correct = np.logical_and.reduce([corrects[name] for name in corrects])
cnn_right_others_wrong = np.logical_and(corrects['CNN'], ~np.logical_and.reduce([corrects[m] for m in ['KNN','DecisionTree','RandomForest','NaiveBayes','MLP']]))
all_wrong = np.logical_and.reduce([~corrects[name] for name in corrects])
idxs_all_correct = pick(all_correct, 6, 123)
idxs_cnn_wins = pick(cnn_right_others_wrong, 6, 456)
idxs_all_wrong = pick(all_wrong, 6, 789)
# Render a few examples where CNN wins
for idx in idxs_cnn_wins[:3]:
    x = X_test[idx].reshape(28,28)
    panels = []
    for name, model in list(models_sklearn.items()) + [('CNN', cnn)]:
        if name == 'NaiveBayes':
            overlay = nb_pixel_contrib(model, x, int(preds[name][idx]))
            pred = int(preds[name][idx]); conf = float(np.max(model.predict_proba(X_test[idx:idx+1])))
            ok = (pred == y_test[idx])
            mask = (overlay >= np.quantile(overlay, 0.90))
        else:
            overlay, pred, base, mask = simple_importance_map_any(model, x, window=4, stride=2, model_name=name)
            conf = base; ok = (pred == y_test[idx])
        panels.append((name, overlay, pred, conf, ok, mask))
    show_row(idx, x, y_test[idx], panels, 'CNN wins example')


In [ ]:
# 10) Quantitative explainability metrics & runtime
import time
from itertools import combinations
import pandas as pd

MODELS_FOR_METRICS = list(models_sklearn.keys()) + ['CNN']
SAMPLE_N = 200  # reduce for faster runs if needed
rng = np.random.RandomState(42)
sample_idxs = rng.choice(X_test.shape[0], size=SAMPLE_N, replace=False)
def compute_metrics_for_models(sample_idxs, window=4, stride=2, top_quantile=0.90):
    mean_drop = {name: [] for name in MODELS_FOR_METRICS}
    masks = {name: [] for name in MODELS_FOR_METRICS}
    times = {name: [] for name in MODELS_FOR_METRICS}
    for idx in sample_idxs:
        x = X_test[idx].reshape(28,28)
        for name in MODELS_FOR_METRICS:
            model = models_sklearn[name] if name!='CNN' else cnn
            t0 = time.perf_counter()
            imp, pred, base, mask = simple_importance_map_any(model, x, window=window, stride=stride, model_name=name, top_quantile=top_quantile)
            t1 = time.perf_counter()
            times[name].append(t1-t0)
            mean_drop[name].append(np.mean(imp))
            masks[name].append(mask.astype(np.uint8))
    mean_drop_agg = {n: float(np.mean(mean_drop[n])) for n in mean_drop}
    mean_time = {n: float(np.mean(times[n])) for n in times}
    pairwise_iou = {}
    for a,b in combinations(MODELS_FOR_METRICS,2):
        vals=[]
        for ma,mb in zip(masks[a],masks[b]):
            inter = np.logical_and(ma,mb).sum(); union = np.logical_or(ma,mb).sum()
            vals.append(inter/union if union>0 else 0.0)
        pairwise_iou[(a,b)] = float(np.mean(vals))
    return mean_drop_agg, mean_time, pairwise_iou
print('Running metrics (≈1 min on GPU, more on CPU)...')
mdrop, mt, p_iou = compute_metrics_for_models(sample_idxs, window=4, stride=2, top_quantile=0.90)
print('\nMean importance per model:')
for k,v in mdrop.items(): print(f'{k:<12}: {v:.6f}')
print('\nMean time per sample (sec):')
for k,v in mt.items(): print(f'{k:<12}: {v:.4f}')
labels = MODELS_FOR_METRICS
mat = pd.DataFrame(np.zeros((len(labels),len(labels))), index=labels, columns=labels)
for (a,b), val in p_iou.items():
    mat.loc[a,b] = val; mat.loc[b,a] = val
print('\nPairwise mask IoU (mean over samples):')
print(mat)
print('\nSensitivity check (short):')
for w,s in [(4,2),(6,3)]:
    md, mtm, _ = compute_metrics_for_models(sample_idxs[:100], window=w, stride=s)
    print(f'window={w}, stride={s} -> CNN mean importance={md["CNN"]:.6f}, time/sample={mtm["CNN"]:.4f}')


In [ ]:
# 11) Grad-CAM example for CNN (contrast with occlusion)
def grad_cam(model, img, last_conv_layer_name='conv2'):
    # img: 2D 28x28
    img_tensor = tf.expand_dims(img.reshape(28,28,1).astype('float32'), axis=0)
    grad_model = tf.keras.models.Model([model.inputs], [model.get_layer(last_conv_layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_tensor)
        pred_index = tf.argmax(predictions[0])
        loss = predictions[:, pred_index]
    grads = tape.gradient(loss, conv_outputs)[0]
    pooled_grads = tf.reduce_mean(grads, axis=(0,1))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(tf.multiply(conv_outputs, pooled_grads), axis=-1).numpy()
    heatmap = np.maximum(heatmap, 0)
    if heatmap.max() != 0:
        heatmap /= heatmap.max()
    heatmap_resized = tf.image.resize(heatmap[..., np.newaxis], (28,28)).numpy().squeeze()
    return heatmap_resized

# Compare occlusion vs Grad-CAM for a random sample
rng = np.random.RandomState(42)
idx = int(rng.choice(X_test.shape[0], size=1)[0])
img = X_test[idx].reshape(28,28)
occl, pred, base, mask = simple_importance_map_any(cnn, img, window=4, stride=2, model_name='CNN', top_quantile=0.90)
gcam = grad_cam(cnn, img, last_conv_layer_name='conv2')
plt.figure(figsize=(9,3))
plt.subplot(1,3,1); plt.imshow(img, cmap='gray'); plt.title(f'Orig True: {y_test[idx]}'); plt.axis('off')
plt.subplot(1,3,2); plt.imshow(img, cmap='gray', alpha=0.8); plt.imshow(occl, cmap='inferno', alpha=0.6); plt.title('Occlusion'); plt.axis('off')
plt.subplot(1,3,3); plt.imshow(img, cmap='gray', alpha=0.8); plt.imshow(gcam, cmap='viridis', alpha=0.6); plt.title('Grad-CAM'); plt.axis('off')
plt.tight_layout(); plt.show()


## 12) Wrap-up & Key Takeaways

- **Visual explanations:** Occlusion maps give an intuitive 'brighter = mattered more' picture. They are model-agnostic but computationally expensive.
- **CNN vs others:** CNN tends to show coherent stroke-focused regions; KNN explains via nearest-neighbor diffs. Naive Bayes has a parametric per-pixel contribution map.
- **Quantitative metrics:** We added mean confidence-drop (average occlusion importance) and pairwise IoU across models to provide numeric comparisons, and a small sensitivity check for window/stride.
- **Limitations:** Occlusion is heuristic and slow. Thresholding (top-10%) and window choices are arbitrary but we included a sensitivity check. Results are empirical on MNIST and should not be overgeneralized.


### How to run in Colab
1. Upload this notebook to Google Colab (or open from Drive).
2. Run the cells top-to-bottom. Training the CNN will take time on CPU; use a GPU runtime in Colab (Runtime -> Change runtime type -> GPU) for speed.
3. If runtime is limited, reduce `SAMPLE_N` in the metrics cell (cell 10) to e.g., 50 for faster runs.

If you'd like, I can also trim the notebook to run much faster for a live demo (e.g., reduce sample sizes, skip training and use pretrained weights).